# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier. One example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it'll pull historical data and initialize the database. 

In [ ]:
from EnvironmentData import EnvironmentData 
#envdt = EnvironmentData(CatsUserID = 2496, out_of_scope = ['-80', 'Cryo tank', 'Water'], testing = True)
envdt = EnvironmentData(CatsUserID = 2496, out_of_scope = ['-80', 'Cryo tank', 'Water'])

Gather readings: SensorReadingRh:  54%|████████████████████████████████████████████████████████████████████████████████████                                                                        | 35/65 [00:40<00:34,  1.15s/it]

Detailed information is saved in the log:

In [ ]:
# Detailed info is saved in the log.
with open('data/EnvironmentData.log', 'r') as file:
    print(file.read().splitlines()[:10])

Raw data is now in the database in `data/sensors.parquet`. 

Initially, we leave the data mostly as-is. We'll clean when moving to analytical steps, this preserves the source data so we can always change our mind about how we decide to view it. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [ ]:
import polars
polars.read_parquet('data/sensors.parquet')

# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [ ]:
envdt.get_current_readings()

# Data is read into new-readings folder for consolidation at the end of the day.
import os
filename = os.listdir('data/new-readings')[0]
print(filename)
polars.read_parquet('data/new-readings/' + filename).sample(5)

# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `devices.parquet`: One row per Device, with measurements across columns. Compare to Sensors which has multiple rows per Device, one for each Sensor, with measurements split beween rows. 
* `sensor_info.parquet`: Information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `utc_info.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [ ]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
print(os.listdir('data/new-readings'))

Let's look at the data we have now:

In [ ]:
# Sensors
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensors = polars.read_parquet('data/sensors.parquet')
sensors.head()

In [ ]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type\.
sensors.tail()

In [ ]:
# Devices
devices = polars.read_parquet('data/devices.parquet')
devices.head()

In [ ]:
# Sensor Info
sensor_info = polars.read_parquet('data/sensor_info.parquet')
sensor_info.head()

In [ ]:
# UTC Date/Time Info
utc_info = polars.read_parquet('data/utc_info.parquet').head()
utc_info.head()

Once we are done working with data intake/processing, we close the class to release lgo file connections:

In [ ]:
# When done, close the connection to the logs. 
envdt.close()

# 4 Analytics

Now we are ready to pull data and run analytics. Here are a few examples: 

In [ ]:
# Histogram of temperature. 
import seaborn, duckdb
dt = duckdb.sql("""
    SELECT SensorReadingF 
    FROM read_parquet('data/sensors.parquet') 
    WHERE SensorReadingF is not null and SensorID = 21373;
""")
seaborn.histplot(dt.fetchnumpy())

In [ ]:
# Time series plot. 
dt = duckdb.sql("""
    SELECT SensorReadingUTC, SensorReadingF 
    FROM read_parquet('data/sensors.parquet') 
    WHERE SensorReadingF is not null and SensorID = 21373;
""")
seaborn.lineplot(x = 'SensorReadingUTC', y= 'SensorReadingF', data=dt.fetchnumpy())

In [ ]:
# Time series plot, queried from Devices instead of Sensors. 
# This is the same data, just organized in a different way. 
dt = duckdb.sql("""
    SELECT SensorReadingUTC, SensorReadingF 
    FROM read_parquet('data/devices.parquet') 
    WHERE SensorReadingF is not null and DeviceDevID = 12162;
""")
seaborn.lineplot(x = 'SensorReadingUTC', y= 'SensorReadingF', data=dt.fetchnumpy())